# Lab 3.3 &mdash; Self-Attention by Hand

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Implement a numerically-stable softmax
- Compute scaled dot-product attention: softmax(Q.Kt / sqrt(d)) . V
- See a query 'attend' to the matching key

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Self-attention (Q/K/V)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-03")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
**Attention** lets each token look at every other token and pull in what's relevant. For queries
**Q**, keys **K** and values **V**:
`attention(Q, K, V) = softmax( Q . Kt / sqrt(d) ) . V`.
The `softmax` turns similarity scores into weights that sum to 1; the scaling by `sqrt(d)` keeps them
stable. This is the exact maths a real model runs &mdash; here we compute it by hand so the mechanism
is unambiguous (Lab 3.9 pulls the same weights out of a real model).

> **See it live (interactive):** [softmax &mdash; from scores to attention weights](../../presentation/softmax-attention-weights.html) &mdash; drag the raw scores and the temperature and watch `exp` &rarr; `normalise` happen. Its defaults reproduce **cat**'s attention row from this lab (the `/sqrt(d)` scaling is the temperature).

## Build it
Implement `softmax` and `attention`.

In [ ]:
import numpy as np
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # numerical stability
    e = ___   # TODO: elementwise exp of x
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d = Q.shape[-1]
    scores = ___   # TODO: Q . K^T divided by sqrt(d)
    weights = softmax(scores, axis=-1)
    return ___   # TODO: weights . V

## Run it for real
Run attention on a tiny example where the answer is obvious.

In [ ]:
try:
    import numpy as np
    Q = np.array([[10.0, 0.0], [0.0, 10.0]])   # two strong queries
    K = np.array([[1.0, 0.0], [0.0, 1.0]])     # two orthogonal keys
    V = np.array([[1.0, 0.0], [0.0, 1.0]])     # values to fetch
    print("attention output:\n", np.round(attention(Q, K, V), 3))
    print("softmax([2,1,0]) =", np.round(softmax(np.array([2.0, 1.0, 0.0])), 3), "(sums to 1)")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- The output is (almost) the identity: query 1 pulls value 1, query 2 pulls value 2 &mdash; each **strong query attended to its matching key**.
- Softmax weights are a **probability distribution** &mdash; they sum to 1, so attention is a weighted average of the values.
- Drop the `/ sqrt(d)` scaling and, for large `d`, the softmax saturates (one weight ~1, the rest ~0) &mdash; that is the vanishing-gradient problem the scaling fixes.

## Run it for real
Identity matrices proved the maths. Now run the **same** `attention` on a **real sentence** &mdash; four words, each given a hand-made 2-D meaning-vector (axis&nbsp;0&nbsp;=&nbsp;`animal`-ness, axis&nbsp;1&nbsp;=&nbsp;`action`-ness). Watch which words attend to which &mdash; this is slide 8's &ldquo;it&nbsp;&rarr;&nbsp;animal&rdquo;, computed.

In [ ]:
try:
    import numpy as np
    # A tiny sentence -- each word gets a hand-made 2-D "meaning" vector.
    words = ["cat", "and", "dog", "ran"]
    E = np.array([[1.0, 0.0],    # cat  -> animal
                  [0.0, 0.0],    # and  -> neutral (no direction)
                  [0.9, 0.0],    # dog  -> animal
                  [0.0, 1.0]])   # ran  -> action

    # Self-attention: every word is its OWN Query, Key and Value.
    W = softmax(E @ E.T / np.sqrt(2), axis=-1)   # row i = how much word i attends to each word

    print("attention weights  (each row sums to 1):")
    print("       " + " ".join(f"{w:>5}" for w in words))
    for w, row in zip(words, W):
        print(f"{w:>5} |" + " ".join(f"{p:5.2f}" for p in row))

    print()
    print("context-aware outputs (each word = weighted blend of the Values):")
    print(np.round(attention(E, E, E), 2))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- Read **cat&rsquo;s row**: most of its weight lands on **cat** and **dog** &mdash; the two animal words (~0.66 combined) &mdash; and little on **and**/**ran**. Attention *pooled the related words*, exactly like slide 8&rsquo;s &ldquo;it &rarr; animal&rdquo;.
- **ran&rsquo;s row** attends mostly to **itself** &mdash; no other word points in the `action` direction.
- **and&rsquo;s row** is split ~evenly: its vector is all-zeros, so it has **no direction** to prefer anyone &mdash; a neutral word attends to everything equally.
- Nothing is hand-waved: we *chose* the vectors, so every weight is explainable. A real model **learns** vectors that do this on real sentences &mdash; Lab 3.9 pulls exactly these weights out of a live model.

## Your turn (open task &mdash; no grader)
Grow the sentence: add a fifth word **&ldquo;kitten&rdquo;** as another animal &mdash; a vector
near **cat**, e.g. `[0.95, 0.05]` &mdash; and rebuild the weight matrix. **Before you run it**, predict:
whose rows change, and where does **cat**&rsquo;s attention go now that there are *three* animals? Then check.
Stretch: flip **ran** toward the `animal` axis and watch the whole matrix rebalance. A "good" answer: you can
call the rough shape of each row before running &mdash; **kitten**&rsquo;s row should come out almost identical
to **cat**&rsquo;s.

---
### Key takeaway
That is the whole mechanism. A transformer stacks many of these attention steps -- and that is what 'Attention Is All You Need' meant.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>